# AutoGen Agent Chain for Automated Code Testing, Review and Debugging

**Objective:** Build an agent chain that does code QA automatically. A **CodeGen** agent writes a date parser, a **Tester** agent writes `pytest` tests for it, and the tests run in a separate process with **coverage** measured. When tests fail, a **Reviewer** agent first decides whether each failing test is wrong or the code is, wrong tests are set aside, and a **Debugger** agent fixes the code. Once everything passes, the Tester gets the **coverage report** and adds tests for code that was never run.

```
CodeGen ──► solution.py ──────┐
Tester  ──► test_solution.py ─┴─► pytest + coverage
                                    │
             ┌──── tests fail ──────┤
             ▼                      │ all pass
        Reviewer: wrong test        ▼
        or wrong code?         coverage below target?
          │          │            │ yes          │ no
   set aside the     │            ▼              ▼
   wrong tests       ▼        Tester adds    ✅ done, save
          └──► Debugger fixes   tests for       artifacts
               solution.py      missed lines
                    └────── run again ◄──────────┘
```

**Tools:** Python (Google Colab), AutoGen AgentChat, OpenAI, pytest, pytest-cov · **Prerequisite:** an OpenAI API key

> **How to run:** add your OpenAI key to Colab **Secrets** as `OPENAI_API_KEY`, then choose **Runtime → Run all**.

## Step 1: Install the required libraries

> **Expected warning:** pip may print `ERROR: pip's dependency resolver...` with a list of `protobuf` conflicts. AutoGen needs protobuf 5.x, while some preinstalled Colab packages (TensorFlow, Google Cloud libraries) want 6.x. This notebook doesn't use those packages, so it's safe to ignore. If a later cell fails to import, choose **Runtime → Restart session** and run all cells again.

In [1]:
!pip install -q -U "autogen-agentchat" "autogen-ext[openai]" pytest pytest-cov

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
googleapis-common-protos 1.76.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.6 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpcio

## Step 2: Import the libraries and load the API key

Don't paste the key into the notebook. Store it in Colab's **Secrets** panel (🔑 icon in the left sidebar) as `OPENAI_API_KEY` and turn on *Notebook access*. If no secret is found, the cell asks for the key in a hidden input box.

In [2]:
import ast
import datetime
import importlib.util
import json
import os
import pathlib
import re
import subprocess
import sys
import tempfile
import textwrap
from getpass import getpass
from typing import Dict, List, Optional

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient

# Load the API key from Colab Secrets, falling back to a hidden prompt
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    api_key = getpass("Enter your OPENAI_API_KEY: ")

os.environ["OPENAI_API_KEY"] = api_key
print("API key loaded ✅" if os.environ.get("OPENAI_API_KEY") else "API key missing ❌")

API key loaded ✅


## Step 3: Configure the model client

All four agents share one OpenAI model client, unless you give the Reviewer a stronger model with `REVIEWER_MODEL`. `temperature=0` keeps the generated code, tests and verdicts consistent from run to run.

In [3]:
MODEL = "gpt-4o-mini"

REVIEWER_MODEL = MODEL   # the Reviewer's judgement matters most: try "gpt-4o" here for a stronger reviewer

model_client = OpenAIChatCompletionClient(
    model=MODEL,
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)
reviewer_client = model_client if REVIEWER_MODEL == MODEL else OpenAIChatCompletionClient(
    model=REVIEWER_MODEL,
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)
print(f"Model client ready: {MODEL} (Reviewer: {REVIEWER_MODEL})")

Model client ready: gpt-4o-mini (Reviewer: gpt-4o-mini)


## Step 4: Settings and run logging

Every run gets its own timestamped folder under `runs/`. Each step of the chain (agent replies, test runs, verdicts, fixes, coverage) is written to `log.jsonl` in that folder, so every run leaves a full audit trail.

In [4]:
MAX_DEBUG_ROUNDS = 3            # Debugger attempts before giving up
MAX_COVERAGE_ROUNDS = 2         # times the Tester may add tests for uncovered code
COVERAGE_TARGET = 95.0          # % of statements and branches in solution.py that the tests must run
MAX_QUARANTINE_FRACTION = 0.3   # the Reviewer may set aside at most 30% of all tests
TEST_TIMEOUT_SEC = 30           # kill a test run that hangs
MAX_LOG_CHARS = 6000            # trim long pytest output before sending it to an agent

COVERAGE_ENABLED = importlib.util.find_spec("pytest_cov") is not None
COV_ARGS = ["--cov=solution", "--cov-branch", "--cov-report=json:cov.json", "--cov-report=term-missing"]
print("Coverage measurement:", "on" if COVERAGE_ENABLED else "OFF (pytest-cov not installed)")


def new_run_dir(label: str) -> pathlib.Path:
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = pathlib.Path("runs") / f"{stamp}_{label}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


def log_event(run_dir: pathlib.Path, kind: str, payload: dict):
    rec = {"ts": datetime.datetime.now(datetime.timezone.utc).isoformat(), "kind": kind, **payload}
    with (run_dir / "log.jsonl").open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")

Coverage measurement: on


## Step 5: Run the tests with pytest and coverage

`run_py_tests` writes `solution.py` and `test_solution.py` to a temporary folder and runs **pytest** in a **separate Python process** with a timeout, so generated code can't hang or crash the notebook. It returns a structured report:

- how many tests passed and failed, and the names of the failing tests
- the **coverage** of `solution.py` (statements and branches), plus the exact lines and branches the tests never ran

If the test file defines no tests at all, pytest reports "no tests ran" and this counts as a failure, so an empty test file can't "pass".

In [5]:
def read_coverage(folder: str) -> Optional[Dict]:
    """Read pytest-cov's JSON report and return coverage details for solution.py"""
    path = pathlib.Path(folder, "cov.json")
    if not path.exists():
        return None
    data = json.loads(path.read_text(encoding="utf-8"))
    for name, info in data.get("files", {}).items():
        if pathlib.Path(name).name == "solution.py":
            return {
                "percent": round(info["summary"]["percent_covered"], 1),
                "missing_lines": info.get("missing_lines", []),
                "missing_branches": info.get("missing_branches", []),
            }
    return None


def run_py_tests(solution_code: str, test_code: str, timeout_sec: int = TEST_TIMEOUT_SEC) -> Dict:
    """Run pytest (with coverage, if available) against the solution in an isolated subprocess"""
    with tempfile.TemporaryDirectory() as tmp:
        pathlib.Path(tmp, "solution.py").write_text(solution_code + "\n", encoding="utf-8")
        pathlib.Path(tmp, "test_solution.py").write_text(test_code + "\n", encoding="utf-8")
        cmd = [sys.executable, "-m", "pytest", "test_solution.py", "-q", "-rfE",
               "--tb=short", "-p", "no:cacheprovider"]
        if COVERAGE_ENABLED:
            cmd += COV_ARGS
        try:
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_sec, cwd=tmp)
        except subprocess.TimeoutExpired:
            return {"passed": False, "n_passed": 0, "n_failed": 0, "failing": [], "coverage": None,
                    "output": f"TIMEOUT: tests did not finish within {timeout_sec}s (possible infinite loop)"}
        output = ((proc.stdout or "") + (proc.stderr or "")).strip()
        coverage = read_coverage(tmp) if COVERAGE_ENABLED else None

    # Read the counts from pytest's final summary line, e.g. "2 failed, 7 passed in 0.31s"
    summary_line = next((line for line in reversed(output.splitlines())
                         if re.search(r"\b(passed|failed|errors?|no tests ran)\b.* in [\d.]+s", line)), "")

    def count(word):
        m = re.search(rf"(\d+) {word}", summary_line)
        return int(m.group(1)) if m else 0

    n_passed, n_failed, n_errors = count("passed"), count("failed"), count("error")
    failing = sorted(set(re.findall(r"^(?:FAILED|ERROR) test_solution\.py::(\w+)", output, re.MULTILINE)))
    if proc.returncode == 5:
        output = "NO TESTS FOUND: test_solution.py defines no test_* functions\n" + output
    return {
        "passed": proc.returncode == 0,
        "n_passed": n_passed,
        "n_failed": n_failed + n_errors,
        "failing": failing,
        "coverage": coverage,
        "output": output,
    }


def score(report: Dict) -> str:
    total = report["n_passed"] + report["n_failed"]
    return f"{report['n_passed']}/{total}" if total else "n/a"


def show(report: Dict, heading: str):
    cov = report["coverage"]
    cov_text = f"{cov['percent']}%" if cov else "n/a"
    print(f"\n[Test Runner] {heading}: passed={report['passed']}  tests={score(report)}  coverage={cov_text}")
    if not report["passed"]:
        print(report["output"][-MAX_LOG_CHARS:])


def failure_details(output: str) -> Dict[str, str]:
    """Split pytest's --tb=short output into {test name: that test's error section}"""
    parts = re.split(r"^_{3,} (\w+) _{3,}$", output, flags=re.MULTILINE)
    return {parts[i]: parts[i + 1].split("=" * 10)[0].strip()[:1500] for i in range(1, len(parts) - 1, 2)}

## Step 6: Helpers for agent replies and test files

- **`extract_code_block`** pulls the Python code out of an agent's reply.
- **`extract_json`** pulls the Reviewer's JSON verdict out of its reply.
- **`remove_tests`** sets tests aside by cutting only those test functions out of the file, keeping everything else, comments included (the removed tests are saved separately).
- **`merge_tests`** appends the Tester's extra tests, renaming any function whose name is already used, so an old test can't be silently overwritten.
- **`ask_agent`** sends one message to an agent and returns its text reply.

In [6]:
CODE_BLOCK_RE = re.compile(r"```(?:python|py)?\s*(.*?)```", re.DOTALL | re.IGNORECASE)


def extract_code_block(text: str) -> str:
    """Return the first fenced code block in a reply, or the whole reply if there isn't one"""
    m = CODE_BLOCK_RE.search(text or "")
    return textwrap.dedent(m.group(1) if m else (text or "")).strip()


def extract_json(text: str) -> Dict:
    """Return the JSON object in a reply (fenced or bare); {} if there isn't a valid one"""
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text or "", re.DOTALL) or re.search(r"(\{.*\})", text or "", re.DOTALL)
    try:
        return json.loads(m.group(1)) if m else {}
    except json.JSONDecodeError:
        return {}


def test_names(test_code: str) -> List[str]:
    tree = ast.parse(test_code)
    return [n.name for n in tree.body if isinstance(n, ast.FunctionDef) and n.name.startswith("test")]


def test_source(test_code: str, name: str) -> str:
    """Return the source of one test function (with decorators)"""
    lines = test_code.splitlines()
    for node in ast.parse(test_code).body:
        if isinstance(node, ast.FunctionDef) and node.name == name:
            start = min([node.lineno] + [d.lineno for d in node.decorator_list])
            return "\n".join(lines[start - 1:node.end_lineno])
    return "(source not found)"


def remove_tests(test_code: str, names: List[str]) -> tuple:
    """Cut the named test functions out of the file, keeping everything else (comments included).
    Return (remaining code, removed code)."""
    lines = test_code.splitlines()
    spans = []
    for node in ast.parse(test_code).body:
        if isinstance(node, ast.FunctionDef) and node.name in names:
            start = min([node.lineno] + [d.lineno for d in node.decorator_list])  # include @decorators
            spans.append((start - 1, node.end_lineno))
    removed, keep = [], set(range(len(lines)))
    for a, b in spans:
        removed.append("\n".join(lines[a:b]))
        keep -= set(range(a, b))
    remaining = "\n".join(lines[i] for i in sorted(keep))
    return re.sub(r"\n{4,}", "\n\n\n", remaining).strip(), "\n\n\n".join(removed)


def merge_tests(test_code: str, extra_code: str) -> tuple:
    """Append extra tests, renaming any that clash with existing names; return (code, new names)"""
    existing = set(test_names(test_code))
    tree = ast.parse(extra_code)
    added = []
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name.startswith("test"):
            base, n = node.name, 2
            while node.name in existing:
                node.name = f"{base}_{n}"
                n += 1
            existing.add(node.name)
            added.append(node.name)
    return test_code.rstrip() + "\n\n\n# --- Added by Tester to raise coverage ---\n" + ast.unparse(tree), added


def numbered(code_text: str) -> str:
    return "\n".join(f"{i:>3}: {line}" for i, line in enumerate(code_text.splitlines(), 1))


async def ask_agent(agent: AssistantAgent, message: str, run_dir: pathlib.Path, label: str) -> str:
    """Send one message to an agent and return its text reply"""
    response = await agent.on_messages(
        [TextMessage(content=message, source="user")],
        cancellation_token=CancellationToken(),
    )
    content = getattr(response.chat_message, "content", "") or ""
    log_event(run_dir, "llm_reply", {"agent": agent.name, "label": label, "content": content})
    return content

## Step 7: Define the task and the four agents

The function to build is described once in `TASK`, and every agent's instructions are built from it. This version uses a date parser, which is much harder to get right than a palindrome check. It has four input formats, case-insensitive month names, leap years, and invalid dates like 30 February that must be rejected.

| Agent | Role |
|---|---|
| **CodeGen** | Writes `solution.py` from the specification |
| **Tester** | Writes `pytest` tests from the specification, then adds tests for any code the coverage report shows was never run |
| **Reviewer** | When tests fail, works out the correct answer for each failing test from the specification, then decides whether **the test is wrong** or **the code is wrong** |
| **Debugger** | Fixes `solution.py` so the remaining tests pass |

In [7]:
TASK = {
    "function_name": "parse_date",
    "signature": "def parse_date(text: str) -> datetime.date",
    "requirements": [
        "Accept exactly these four formats; leading and trailing whitespace is ignored:",
        "  1. ISO: YYYY-MM-DD, e.g. '2026-11-05'",
        "  2. US numeric: MM/DD/YYYY with 1 or 2 digit month and day, e.g. '11/05/2026' or '1/5/2026' (month comes FIRST)",
        "  3. Month name first: 'Month D, YYYY', e.g. 'November 5, 2026' or 'Nov 5, 2026'",
        "  4. Day first: 'D Month YYYY', e.g. '5 November 2026' or '5 Nov 2026'",
        "Month names may be full ('September') or the standard 3-letter abbreviation ('Sep'), in any letter case.",
        "Return a datetime.date object.",
        "Raise ValueError for a date that doesn't exist (e.g. '2026-02-30', or Feb 29 in a non-leap year).",
        "Raise ValueError for an empty string or any text that doesn't match one of the four formats.",
        "Raise TypeError if text is not a str.",
    ],
    "edge_cases": "each of the four formats, single-digit day and month, mixed-case month names, abbreviations, "
                  "surrounding whitespace, leap years (2024-02-29 valid, 2025-02-29 invalid), invalid month or day, "
                  "empty string, unsupported formats, non-string input",
}

spec_text = "\n".join(f"- {r}" for r in TASK["requirements"])

codegen = AssistantAgent(
    name="CodeGen",
    model_client=model_client,
    system_message=(
        "You are CodeGen. Write Python code for solution.py containing a SINGLE public function:\n"
        f"{TASK['signature']}\nRequirements:\n{spec_text}\n"
        "- Use only the Python standard library. Do not print.\n"
        "Reply with ONLY a Python fenced code block for solution.py."
    ),
)

tester = AssistantAgent(
    name="Tester",
    model_client=model_client,
    system_message=(
        "You are Tester. Write pytest tests in test_solution.py.\n"
        f"Start with: import pytest, from datetime import date, and from solution import {TASK['function_name']}\n"
        f"The function under test is {TASK['signature']} with these requirements:\n{spec_text}\n"
        f"Cover typical and edge cases ({TASK['edge_cases']}).\n"
        "Put each behaviour in its own function named test_..., using plain assert and pytest.raises.\n"
        "Only test behaviour the requirements actually state. Do not invent extra rules.\n"
        "When asked to improve coverage, reply with ONLY the NEW test functions (plus any imports they need), "
        "using names not already in the file.\n"
        "Always reply with ONLY a Python fenced code block."
    ),
)

reviewer = AssistantAgent(
    name="Reviewer",
    model_client=reviewer_client,
    system_message=(
        "You are Reviewer, an impartial QA lead. Some tests are failing. Judge EACH failing test on its own, "
        "using ONLY the requirements below, never what the current code does.\n"
        "For each test, step by step:\n"
        "1. spec_says: apply the requirements yourself to the test's input and write the correct result "
        "(a value, or the exception that must be raised).\n"
        "2. test_expects: write what the test asserts.\n"
        "3. verdict: if spec_says and test_expects differ, the verdict is WRONG_TEST. If they match, the verdict "
        "is CODE_BUG. Only if the requirements genuinely don't determine the result, choose CODE_BUG.\n"
        f"Requirements for {TASK['signature']}:\n{spec_text}\n"
        'Reply with ONLY a JSON object: {"verdicts": [{"test": "<test name>", "spec_says": "...", '
        '"test_expects": "...", "verdict": "WRONG_TEST" or "CODE_BUG", "reason": "<one sentence>"}]}'
    ),
)

debugger = AssistantAgent(
    name="Debugger",
    model_client=model_client,
    system_message=(
        "You are Debugger. Given failing pytest output, the test file and the current solution.py, "
        "produce a FIXED solution.py that passes the tests. Keep the same signature:\n"
        f"{TASK['signature']}\nThe requirements are:\n{spec_text}\n"
        "Use only the Python standard library.\n"
        "Reply with ONLY a Python fenced code block for solution.py."
    ),
)

AGENTS = [codegen, tester, reviewer, debugger]
print("Agents ready:", ", ".join(a.name for a in AGENTS))

Agents ready: CodeGen, Tester, Reviewer, Debugger


## Step 8: The Reviewer step, with guardrails

`review_failures` asks the Reviewer about every failing test. Each review starts with a **fresh conversation**, so the Reviewer isn't anchored to its earlier verdicts. It sees each failing test **one by one**, with that test's source code and its own pytest error, and must **work out the correct answer from the specification first**, then compare it with what the test expects. The Reviewer doesn't see `solution.py`, so it can't be swayed by what the code happens to do.

Because the Reviewer is also a language model, its verdicts go through a few guardrails before any test is removed:

- **Only failing tests** can be set aside. A verdict about a test that passed is ignored.
- **A cap.** Across the whole run, at most `MAX_QUARANTINE_FRACTION` of the original tests can be set aside. A verdict that would go over the limit is rejected entirely. Wholesale "the tests are wrong" is more likely to mean the code is wrong.
- **Nothing is deleted.** Removed tests are saved to `quarantined_tests.py` in the run folder with the Reviewer's reasons, so a person can check them.

In [8]:
async def review_failures(report: Dict, solution: str, tests: str, run_dir: pathlib.Path, label: str,
                          budget: int):
    """Return (tests with wrong tests removed, list of {test, reason} that were set aside).
    budget = how many more tests may be set aside in this run."""
    await reviewer.on_reset(CancellationToken())  # judge fresh each time, not anchored to earlier verdicts
    details = failure_details(report["output"])
    sections = [
        f"### {name}\n```python\n{test_source(tests, name)}\n```\n"
        f"pytest error:\n```\n{details.get(name, '(no details)')}\n```"
        for name in report["failing"]
    ]
    prompt = (
        f"{len(report['failing'])} test(s) are failing. Judge each one separately.\n\n"
        + "\n\n".join(sections)
        + "\n\nShared setup at the top of test_solution.py:\n```python\n"
        + "\n".join(l for l in tests.splitlines() if l.startswith(("import ", "from ")))
        + "\n```"
    )
    verdicts = extract_json(await ask_agent(reviewer, prompt, run_dir, label)).get("verdicts", [])
    wrong = [v for v in verdicts if v.get("verdict") == "WRONG_TEST" and v.get("test") in report["failing"]]

    for v in verdicts:
        print(f"[Reviewer] {v.get('test')}: {v.get('verdict')}  (spec says: {v.get('spec_says')} | "
              f"test expects: {v.get('test_expects')})")
    log_event(run_dir, "review", {"label": label, "verdicts": verdicts})

    if not wrong:
        return tests, []
    if len(wrong) > budget:
        print(f"[Reviewer] Verdict rejected: wants to set aside {len(wrong)} test(s), but only {budget} more "
              f"may be set aside in this run (limit {MAX_QUARANTINE_FRACTION:.0%} of the tests). "
              "Treating all failures as code bugs.")
        log_event(run_dir, "review_rejected", {"label": label, "wrong": len(wrong), "budget": budget})
        return tests, []

    remaining, removed_code = remove_tests(tests, [v["test"] for v in wrong])
    with (run_dir / "quarantined_tests.py").open("a", encoding="utf-8") as f:
        for v in wrong:
            f.write(f"# {v['test']}: {v.get('reason', '')}\n")
        f.write(removed_code + "\n\n\n")
    print(f"[Reviewer] Set aside {len(wrong)} wrong test(s): {', '.join(v['test'] for v in wrong)}")
    log_event(run_dir, "quarantine", {"label": label, "tests": wrong})
    return remaining, wrong

## Step 9: The orchestrator

`run_chain()` runs the whole loop and returns a summary. It can also start from code and tests you supply, which is how Step 11 shows the Reviewer and Debugger at work.

1. **CodeGen** writes `solution.py` and the **Tester** writes `test_solution.py` (unless they are supplied).
2. The tests run with pytest and coverage.
3. **If tests fail:** the **Reviewer** judges each failing test, and wrong tests are set aside. If failures remain, the **Debugger** fixes the code (up to `MAX_DEBUG_ROUNDS`).
4. **If all tests pass but coverage is below `COVERAGE_TARGET`:** the **Tester** gets the code with line numbers and the exact lines and branches never run, and adds tests for them (up to `MAX_COVERAGE_ROUNDS`). The new tests go through step 3 if they fail.
5. The final code, tests, any set-aside tests and the log are saved to the run's folder.

In [9]:
async def run_chain(label: str = "generate",
                    starting_solution: Optional[str] = None,
                    starting_tests: Optional[str] = None) -> Dict:
    run_dir = new_run_dir(label)
    for agent in AGENTS:  # each run starts with a clean conversation history
        await agent.on_reset(CancellationToken())

    print(f"== AutoGen Code Testing, Review and Debugging Chain ({MODEL}) ==")
    log_event(run_dir, "start", {"model": MODEL, "label": label, "task": TASK["signature"]})

    # 1) CodeGen creates solution.py
    if starting_solution is None:
        reply = await ask_agent(codegen, "Write solution.py as specified. Return ONLY the code block.", run_dir, "codegen")
        solution = extract_code_block(reply)
        if f"def {TASK['function_name']}" not in solution:
            reply = await ask_agent(codegen, f"Please include {TASK['signature']}", run_dir, "codegen_retry")
            solution = extract_code_block(reply)
        print("[CodeGen] Generated solution.py")
    else:
        solution = textwrap.dedent(starting_solution).strip()
        print("[Setup] Starting from the provided solution.py")
    log_event(run_dir, "solution", {"code": solution})

    # 1b) Tester creates test_solution.py
    if starting_tests is None:
        reply = await ask_agent(tester, f"Create test_solution.py for {TASK['function_name']}. Return ONLY the code block.", run_dir, "tester")
        tests = extract_code_block(reply)
        print("[Tester] Generated test_solution.py")
    else:
        tests = textwrap.dedent(starting_tests).strip()
        print("[Setup] Using the provided test_solution.py")
    log_event(run_dir, "tests", {"code": tests})

    debug_rounds, coverage_rounds, quarantined = 0, 0, []
    first_report = None
    try:
        original_total = len(test_names(tests))
    except SyntaxError:
        original_total = 0

    while True:
        # 2) Run the tests
        report = run_py_tests(solution, tests)
        first_report = first_report or report
        show(report, "Test run")
        log_event(run_dir, "test_run", {k: v for k, v in report.items()})

        # 3) Failures: Reviewer first, then Debugger
        if not report["passed"]:
            if debug_rounds >= MAX_DEBUG_ROUNDS:
                break
            budget = int(MAX_QUARANTINE_FRACTION * original_total) - len(quarantined)
            if report["failing"] and budget > 0:
                tests, wrong = await review_failures(report, solution, tests, run_dir,
                                                     f"review_{debug_rounds + 1}", budget)
                if wrong:
                    quarantined += wrong
                    continue  # re-run without the wrong tests before changing any code

            debug_rounds += 1
            print(f"\n[Debugger] Fix attempt #{debug_rounds}")
            prompt = (
                "Tests failed with the output below.\n\n"
                f"=== PYTEST OUTPUT ===\n{report['output'][-MAX_LOG_CHARS:]}\n\n"
                f"=== test_solution.py ===\n```python\n{tests}\n```\n\n"
                f"=== CURRENT solution.py ===\n```python\n{solution}\n```"
            )
            fixed = extract_code_block(await ask_agent(debugger, prompt, run_dir, f"debugger_round_{debug_rounds}"))
            if fixed and fixed != solution:
                solution = fixed
            else:
                print("[Debugger] No change proposed")
            log_event(run_dir, "debug_fix", {"round": debug_rounds, "code": solution})
            continue

        # 4) All passing: is coverage high enough?
        cov = report["coverage"]
        if cov and cov["percent"] < COVERAGE_TARGET and coverage_rounds < MAX_COVERAGE_ROUNDS:
            coverage_rounds += 1
            print(f"\n[Tester] Coverage {cov['percent']}% is below {COVERAGE_TARGET}%: adding tests (round {coverage_rounds})")
            prompt = (
                f"All tests pass, but they only cover {cov['percent']}% of solution.py "
                f"(target {COVERAGE_TARGET}%).\n"
                f"Lines never run: {cov['missing_lines']}\n"
                f"Branches never taken (from line -> to line; negative means 'exit function'): {cov['missing_branches']}\n\n"
                f"=== solution.py (with line numbers) ===\n{numbered(solution)}\n\n"
                f"Existing test names: {', '.join(test_names(tests))}\n\n"
                "Write NEW test functions that exercise those lines and branches. "
                "Each test must still follow the requirements; never assert behaviour the requirements don't state."
            )
            extra = extract_code_block(await ask_agent(tester, prompt, run_dir, f"tester_coverage_{coverage_rounds}"))
            try:
                tests, added = merge_tests(tests, extra)
                print(f"[Tester] Added {len(added)} test(s): {', '.join(added)}")
                log_event(run_dir, "coverage_tests", {"round": coverage_rounds, "added": added, "code": extra})
            except SyntaxError as e:
                print(f"[Tester] Extra tests had a syntax error and were skipped: {e}")
                log_event(run_dir, "coverage_tests_invalid", {"round": coverage_rounds, "error": str(e)})
            continue
        break

    # 5) Save artifacts and finish
    (run_dir / "solution.py").write_text(solution + "\n", encoding="utf-8")
    (run_dir / "test_solution.py").write_text(tests + "\n", encoding="utf-8")
    status = "ALL TESTS PASSED" if report["passed"] else "STOPPED WITH FAILURES (see log.jsonl)"
    cov = report["coverage"]
    summary = {
        "label": label,
        "passed": report["passed"],
        "first_score": score(first_report),
        "score": score(report),
        "coverage": f"{cov['percent']}%" if cov else "n/a",
        "first_coverage": f"{first_report['coverage']['percent']}%" if first_report["coverage"] else "n/a",
        "debug_rounds": debug_rounds,
        "coverage_rounds": coverage_rounds,
        "quarantined": [q["test"] for q in quarantined],
        "still_failing": report["failing"],
        "solution": solution,
        "tests": tests,
        "run_dir": run_dir,
    }
    print(f"\n== RESULT: {status}  |  tests: {summary['score']}  |  coverage: {summary['coverage']}  |  "
          f"debug rounds: {debug_rounds}  |  coverage rounds: {coverage_rounds}  |  tests set aside: {len(quarantined)}")
    print(f"Artifacts saved to: {run_dir}")
    log_event(run_dir, "end", {k: v for k, v in summary.items() if k not in ("solution", "tests")})
    return summary

## Step 10: Run the chain

CodeGen writes the parser, Tester writes the tests, and the loop runs until the tests pass with enough coverage or a limit is reached. A date parser has many edge cases, so it's normal to see a Reviewer verdict, a Debugger round or a coverage round here.

In [10]:
result = await run_chain("generate")

== AutoGen Code Testing, Review and Debugging Chain (gpt-4o-mini) ==
[CodeGen] Generated solution.py
[Tester] Generated test_solution.py

[Test Runner] Test run: passed=False  tests=7/9  coverage=100.0%
..FF.....                                                                [100%]
=================================== FAILURES ===================================
_________________________ test_parse_month_name_first __________________________
test_solution.py:14: in test_parse_month_name_first
    assert parse_date('Nov 5, 2026') == date(2026, 11, 5)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
solution.py:31: in parse_date
    month = datetime.datetime.strptime(month_str, '%B').month
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
/usr/lib/python3.13/_strptime.py:789: in _strptime_datetime
    tt, fraction, gmtoff_fraction = _strptime(data_string, format)
                                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
/usr/lib/python3.13/_strptime.py:555: in _strptime
    raise Value

## Step 11: Watch the Reviewer and Debugger handle broken code *and* a broken test

This run starts from two deliberate problems:

1. **Buggy code:** the starting `solution.py` only understands ISO dates (`2026-11-05`) and doesn't strip whitespace, so most format tests fail.
2. **A wrong test:** one extra test is added that reads `03/02/2026` as **3 February**. The specification says the month comes first in this format, so the correct answer is **2 March**: the test is wrong, not the code.

It reuses the tests generated in Step 10. A good run should show the Reviewer setting aside `test_slash_date_example`, then the Debugger fixing the code to pass all the real tests.

In [11]:
BUGGY_SOLUTION = """
import datetime

def parse_date(text: str) -> datetime.date:
    year, month, day = text.split("-")
    return datetime.date(int(year), int(month), int(day))
"""

WRONG_TEST = """

def test_slash_date_example():
    assert parse_date("03/02/2026") == date(2026, 2, 3)
"""

demo_tests = result["tests"]
if "from datetime import date" not in demo_tests:
    demo_tests = "from datetime import date\n" + demo_tests

debug_result = await run_chain("review_debug_demo", starting_solution=BUGGY_SOLUTION,
                               starting_tests=demo_tests + WRONG_TEST)

== AutoGen Code Testing, Review and Debugging Chain (gpt-4o-mini) ==
[Setup] Starting from the provided solution.py
[Setup] Using the provided test_solution.py

[Test Runner] Test run: passed=False  tests=5/10  coverage=100.0%
.FFF....FF                                                               [100%]
=================================== FAILURES ===================================
__________________________ test_parse_us_numeric_date __________________________
test_solution.py:9: in test_parse_us_numeric_date
    assert parse_date('11/05/2026') == date(2026, 11, 5)
           ^^^^^^^^^^^^^^^^^^^^^^^^
solution.py:4: in parse_date
    year, month, day = text.split("-")
    ^^^^^^^^^^^^^^^^
E   ValueError: not enough values to unpack (expected 3, got 1)
_________________________ test_parse_month_name_first __________________________
test_solution.py:13: in test_parse_month_name_first
    assert parse_date('November 5, 2026') == date(2026, 11, 5)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^

## Step 12: Summary and final code

In [12]:
print(f"{'Run':<19}{'Passed':<8}{'Tests (first -> final)':<24}{'Coverage (first -> final)':<27}"
      f"{'Debug':<7}{'Cov rounds':<12}{'Set aside':<26}Still failing")
print("-" * 135)
for r in [result, debug_result]:
    print(f"{r['label']:<19}{str(r['passed']):<8}{r['first_score'] + ' -> ' + r['score']:<24}"
          f"{r['first_coverage'] + ' -> ' + r['coverage']:<27}{r['debug_rounds']:<7}{r['coverage_rounds']:<12}"
          f"{', '.join(r['quarantined']) or '-':<26}{', '.join(r['still_failing']) or '-'}")

print("\n--- Final solution.py from the review/debug run ---")
print(debug_result["solution"])

q = debug_result["run_dir"] / "quarantined_tests.py"
if q.exists():
    print("\n--- Tests the Reviewer set aside ---")
    print(q.read_text())

Run                Passed  Tests (first -> final)  Coverage (first -> final)  Debug  Cov rounds  Set aside                 Still failing
---------------------------------------------------------------------------------------------------------------------------------------
generate           True    7/9 -> 9/9              100.0% -> 100.0%           1      0           -                         -
review_debug_demo  True    5/10 -> 9/9             100.0% -> 100.0%           2      0           test_slash_date_example   -

--- Final solution.py from the review/debug run ---
import datetime
import re

def parse_date(text: str) -> datetime.date:
    if not isinstance(text, str):
        raise TypeError("Input must be a string")

    text = text.strip()

    # ISO format: YYYY-MM-DD
    iso_match = re.match(r'^(\d{4})-(\d{2})-(\d{2})$', text)
    if iso_match:
        year, month, day = map(int, iso_match.groups())
        return datetime.date(year, month, day)

    # US numeric format: MM/DD/

## Conclusion

You've built an AutoGen agent chain that writes code, writes its own tests, runs them with pytest and coverage, separates wrong tests from real bugs, fixes the code, and fills coverage gaps, with every step logged to a timestamped `runs/` folder.

**Ideas to extend it:**
- Let the Tester *rewrite* a test the Reviewer flags as wrong, instead of only setting it aside
- Add mutation testing (for example with `mutmut`) to check that the tests actually catch bugs, not just run lines
- Run several tasks from a list and report a pass rate per model
- Compare models (for example `gpt-4o-mini` vs `gpt-4o`) on rounds needed and final coverage